In [1]:
import os
import pandas as pd

In [2]:
def parse_excel_all_folds(excel_path):
    """
    Parse all_folds.xlsx file where each sheet represents a seed,
    and combine all fold data into a single DataFrame.
    """
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"File not found: {excel_path}")
    
    try:
        # Read all sheets
        xl = pd.ExcelFile(excel_path, engine='openpyxl')
        all_folds_data = []
        
        for sheet_name in xl.sheet_names:
            df_sheet = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')
            all_folds_data.append(df_sheet)
        
        # Combine all sheets into one DataFrame
        df_combined = pd.concat(all_folds_data, ignore_index=True)
        
        print(f"Successfully loaded data from: {excel_path}")
        print(f"Number of sheets (seeds): {len(xl.sheet_names)}")
        print(f"Combined shape: {df_combined.shape}")
        print(f"Columns: {df_combined.columns.tolist()}")
        print(f"Unique seeds: {df_combined['Seed'].nunique()}")
        print(f"Unique folds: {sorted(df_combined['Fold'].unique())}")
        
        return df_combined
    
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        raise

def merge_dataframes(F5_dir, L5_dir, basename):
    df_f5 = parse_excel_all_folds(F5_dir)
    df_l5 = parse_excel_all_folds(L5_dir)
    df_merged = pd.concat([df_f5, df_l5], ignore_index=True)
    dataset_prefix = F5_dir[3]
    output_dir = f'10S{dataset_prefix}_{basename}'
    os.makedirs(output_dir, exist_ok=True)
    print(f"Merged DataFrame shape: {df_merged.shape}")
    df_merged.to_excel(os.path.join(output_dir, f"10S{dataset_prefix}_{basename}_all_folds.xlsx"), index=False, engine='openpyxl')
    # save summary
    # Calculate summary statistics mean and std_dev
    metrics = [
            'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
            'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)',
            'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)',
            'Min GPU Utilization (%)', 'Max GPU Utilization (%)', 'Avg GPU Utilization (%)'
        ]
    summary_df = pd.DataFrame({
        'Mean': df_merged[metrics].mean(),
        'Std Dev': df_merged[metrics].std()
    })
    with pd.ExcelWriter(os.path.join(output_dir, f"10S{dataset_prefix}_{basename}_summary.xlsx"), engine='openpyxl') as writer:
        # Sheet 1: Overall mean and std dev
        summary_df.to_excel(writer, sheet_name='Overall_Summary', index=True)
    summary_df
    return df_merged, summary_df

In [3]:
df_merged, summary_df = merge_dataframes(
    F5_dir='F5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln/F5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx',
    L5_dir='L5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln/L5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx',
    basename='tcn_model_fine_tune_mb_seqOut_sgd_ln')
df_merged

Successfully loaded data from: F5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln/F5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx
Number of sheets (seeds): 5
Combined shape: (25, 18)
Columns: ['Fold', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)', 'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)', 'Min GPU Utilization (%)', 'Max GPU Utilization (%)', 'Avg GPU Utilization (%)', 'Seed', 'Variant', 'Dataset']
Unique seeds: 5
Unique folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Successfully loaded data from: L5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln/L5ST_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx
Number of sheets (seeds): 5
Combined shape: (25, 18)
Columns: ['Fold', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)', 'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)', 'Min GPU Utilization (%

,Fold,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Training Time (s),Epochs Run,Avg Time/Epoch (s),Min GPU Memory (MB),Max GPU Memory (MB),Avg GPU Memory (MB),Min GPU Utilization (%),Max GPU Utilization (%),Avg GPU Utilization (%),Seed,Variant,Dataset
0,1,0.940789,0.925061,0.941250,0.933086,0.984807,1248.843054,19,65.728582,16615,16615,16615,80,93,90.052632,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
1,2,0.950658,0.940447,0.947500,0.943960,0.987552,741.517135,11,67.410649,16617,16617,16617,88,94,91.272727,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
2,3,0.950110,0.941469,0.945000,0.943231,0.985760,993.868259,15,66.257884,16619,16619,16619,83,93,90.600000,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
3,4,0.936404,0.911942,0.946183,0.928747,0.984232,867.239630,13,66.710741,16621,16621,16621,88,93,91.384615,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
4,5,0.946242,0.923821,0.956195,0.939729,0.988837,1056.401125,16,66.025070,16621,16621,16621,64,92,88.937500,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
5,1,0.952851,0.939655,0.953750,0.946650,0.987747,993.871796,15,66.258120,16623,16623,16623,81,94,90.866667,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
6,2,0.950110,0.951592,0.933750,0.942587,0.986444,805.753728,12,67.146144,16625,16625,16625,90,92,91.333333,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
7,3,0.938048,0.930991,0.927500,0.929242,0.984252,805.913939,12,67.159495,16627,16627,16627,90,93,91.500000,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
8,4,0.953947,0.935445,0.961202,0.948148,0.991180,930.626024,14,66.473287,16627,16627,16627,82,92,89.714286,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter
9,5,0.935820,0.918919,0.936170,0.927464,0.984662,932.583352,14,66.613097,16629,16629,16629,90,94,91.428571,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Twitter


In [4]:
summary_df

,Mean,Std Dev
Accuracy,0.945498,0.005435
Precision,0.939701,0.013971
Recall,0.936093,0.013083
F1-Score,0.937740,0.006076
ROC-AUC,0.986650,0.002179
Training Time (s),936.554199,156.166542
Epochs Run,13.980000,2.478396
Avg Time/Epoch (s),67.102181,0.746807
Min GPU Memory (MB),16633.320000,10.539411
Max GPU Memory (MB),16633.320000,10.539411
